# Sprint 2 - Data Quality, Joins and Outliers**Buy Now, Pay Later merchant ranking | MAST30034 Applied Data Science**This notebook answers the three questions set for the Sprint 2 checkpoint:1. What NULL values appeared after joining the datasets, and what did we do with them?2. Was anything missing that shouldn't have been, once we joined to our external dataset?3. What does the transaction value distribution look like before and after removing outliers?Everything below reads from `data/curated/`, which is produced by running thepipeline in order. Nothing in this notebook transforms data - the transformationslive in `scripts/` so that they run identically whether or not anyone opens a notebook.```python scripts/download_external.py   # ABS datapython scripts/etl_01_raw.py          # typed copies of the source filespython scripts/etl_02_curated.py      # joins + business rulespython scripts/etl_03_features.py     # merchant-level feature table```

In [ ]:
import sys, jsonfrom pathlib import Pathimport matplotlib.pyplot as pltimport numpy as npimport pandas as pdsys.path.insert(0, str(Path.cwd().parent / "scripts"))import configfrom spark_session import create_sparkfrom pyspark.sql import functions as Fspark = create_spark("Sprint 2 - data quality")spark.sparkContext.setLogLevel("ERROR")pd.set_option("display.float_format", lambda v: f"{v:,.2f}")plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True, "grid.alpha": 0.3})

## 1. What we were givenBefore looking for problems it is worth stating plainly what is in the box.

In [ ]:
transactions = spark.read.parquet(str(config.CURATED_DIR / "transactions"))overview = transactions.agg(    F.count("*").alias("transactions"),    F.countDistinct("merchant_abn").alias("distinct merchants"),    F.countDistinct("user_id").alias("distinct customers"),    F.min("order_datetime").alias("first date"),    F.max("order_datetime").alias("last date"),    F.sum("dollar_value").alias("total dollar value"),).toPandas().Toverview.columns = ["value"]overview

### Immediate observation: very few consumers actually transactThe consumer table describes roughly half a million people, but only a smallfraction of them appear in the transaction data at all. This is not a dataerror - it is how the dataset was generated - but it matters for the rankingsystem, because any feature phrased as "share of all consumers" will bemisleadingly tiny. Customer counts are therefore always expressed relative tothe transacting population, never the full consumer table.

In [ ]:
consumers = spark.read.parquet(str(config.RAW_DIR / "consumers"))n_consumers = consumers.count()n_transacting = transactions.select("user_id").distinct().count()print(f"consumers on file:            {n_consumers:,}")print(f"consumers with a transaction: {n_transacting:,}")print(f"share transacting:            {n_transacting / n_consumers:.2%}")

## 2. NULLs after joiningThe joins are all LEFT joins. An inner join would have produced a clean-lookingtable by deleting the very rows that tell us something is wrong, so instead eachjoin carries a boolean flag recording whether it matched.

In [ ]:
join_quality = transactions.agg(    F.count("*").alias("total rows"),    F.sum((~F.col("has_merchant_record")).cast("int")).alias("no merchant record"),    F.sum((~F.col("has_consumer_record")).cast("int")).alias("no consumer record"),).toPandas().Tjoin_quality.columns = ["rows"]join_quality["% of total"] = 100 * join_quality["rows"] / join_quality.loc["total rows", "rows"]join_quality

### The merchant gap is the one that mattersA block of transactions references merchant ABNs that do not appear in`tbl_merchants` at all. These are not nulls in a field - they are entiremerchants we have money flowing through but no record for. Because the merchantrecord is where `take_rate` and `category` come from, we cannot compute eitherthe BNPL firm's revenue or the correct outlier threshold for these rows.Three options were considered:| Option | Consequence ||---|---|| Drop the rows | Loses a meaningful share of total transaction value with no record of it || Impute the take rate from the category average | Impossible - the category is missing too || Keep, flag, exclude from ranking | Value is preserved and auditable; these merchants simply cannot be ranked |We chose the third. These merchants are **not candidates for onboarding** —we have no partnership record for them — so excluding them from the ranking iscorrect on business grounds, not just convenient. The volume is reported belowso the exclusion is visible rather than silent.

In [ ]:
unmatched = (    transactions.filter(~F.col("has_merchant_record"))    .agg(        F.countDistinct("merchant_abn").alias("merchants"),        F.count("*").alias("transactions"),        F.sum("dollar_value").alias("dollar value"),    )    .toPandas())total_value = transactions.agg(F.sum("dollar_value")).collect()[0][0]unmatched["% of all dollars"] = 100 * unmatched["dollar value"] / total_valueunmatched

### Merchants with a record but no transactionsThe opposite case also exists: merchants who signed up but have not transactedin this window. The project spec calls these out directly - *"a new merchantwith little information on how they will perform"*. They are kept and handledseparately in the ranking system rather than being scored as though they hadzero revenue, which would rank them below merchants who are genuinely failing.

In [ ]:
merchants = spark.read.parquet(str(config.RAW_DIR / "merchants"))active = transactions.select("merchant_abn").distinct()no_activity = merchants.join(active, "merchant_abn", "left_anti")print(f"merchants on file with no transactions in this window: {no_activity.count()}")no_activity.select("merchant_name", "category", "revenue_level", "take_rate").limit(10).toPandas()

## 3. Joining to the external ABS dataConsumers are located by postcode; every useful ABS dataset is published by SA2(Statistical Area Level 2). These do not nest — one postcode can span severalSA2s and vice versa — so the ABS postcode-to-SA2 correspondence file is used,weighting each SA2's contribution by the `RATIO_FROM_TO` column.The ABS itself warns that postcode boundaries are not authoritative and shouldnot be relied on for geocoding. **This is a stated assumption of the project,not a solved problem.** Two separate mappings are built for two different uses:- **ratio-weighted postcode attributes** — used for all modelling, because it  avoids duplicating a consumer across several SA2s and inflating their  transaction counts;- **dominant SA2 per postcode** — used only for choropleth maps, where a single  region has to be picked.The coverage check below is the answer to *"was anything missing that shouldn'thave been?"*

In [ ]:
import geotry:    correspondence = geo.load_correspondence()    postcodes = [r.postcode for r in consumers.select("postcode").distinct().collect()]    coverage = geo.coverage_report(postcodes, correspondence)    for key, value in coverage.items():        print(f"{key}: {value}")except FileNotFoundError as error:    print(error)    print("\nRun `python scripts/download_external.py` before this cell.")

## 4. Outliers in transaction value### Before any rules are applied

In [ ]:
values = (    transactions.select("dollar_value")    .sample(fraction=0.15, seed=config.RANDOM_SEED)    .toPandas()["dollar_value"])print(values.describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99, 0.999]).to_string())

Two problems are visible in that summary, at opposite ends of the distribution.**The bottom.** The minimum transaction value is a small fraction of one cent.These cannot be real "pay in 5 instalments" purchases — you cannot split afraction of a cent into five payments. They are an artefact of the datagenerator. Left in, they drag the mean basket size down and make merchants lookcheaper than they are.**The top.** The distribution has a very long right tail. The naive fix — oneglobal dollar cap — is wrong here, because the same figure means differentthings in different categories. Eight thousand dollars is an unremarkablefurniture order and an impossible florist order. The threshold is thereforecomputed **within each merchant category**, at the quantile set in`config.CATEGORY_OUTLIER_QUANTILE`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))axes[0].hist(values, bins=120, range=(0, 1500), color="#4C72B0")axes[0].set(title="Transaction value (linear, capped at $1,500 for display)",            xlabel="dollar value", ylabel="transactions")axes[1].hist(np.log10(values[values > 0]), bins=120, color="#4C72B0")axes[1].axvline(np.log10(config.MIN_TRANSACTION_VALUE), color="crimson",                linestyle="--", label=f"min value rule (${config.MIN_TRANSACTION_VALUE:.2f})")axes[1].set(title="Transaction value (log10)", xlabel="log10(dollar value)")axes[1].legend()plt.tight_layout()plt.show()

The log plot is the informative one: the left-hand lump below the red line isthe block of sub-cent transactions, clearly separated from the main body ofgenuine purchases rather than blending into it. That separation is whatjustifies a hard floor rather than a soft one.

In [ ]:
# Per-category thresholds actually applied by the pipeline.thresholds = (    transactions.filter(F.col("category").isNotNull())    .groupBy("category", "segment")    .agg(        F.first("category_upper_threshold").alias("upper threshold"),        F.expr("percentile_approx(dollar_value, 0.5)").alias("median basket"),        F.count("*").alias("transactions"),    )    .orderBy(F.desc("upper threshold"))    .toPandas())thresholds

The spread across categories is the argument for this approach in one table: thecap for the most expensive category sits far above the cap for the cheapest. Asingle global threshold would simultaneously fail to catch impossibletransactions in low-value categories and discard legitimate ones in high-valuecategories.

In [ ]:
rules = transactions.agg(    F.count("*").alias("total rows"),    F.sum(F.col("below_min_value").cast("int")).alias("below minimum value"),    F.sum(F.col("above_category_threshold").cast("int")).alias("above category threshold"),    F.sum((~F.col("has_merchant_record")).cast("int")).alias("no merchant record"),    F.sum(F.col("is_valid").cast("int")).alias("retained as valid"),).toPandas().Trules.columns = ["rows"]rules["% of total"] = 100 * rules["rows"] / rules.loc["total rows", "rows"]rules

### After the rules

In [ ]:
clean = (    transactions.filter(F.col("is_valid"))    .select("dollar_value")    .sample(fraction=0.15, seed=config.RANDOM_SEED)    .toPandas()["dollar_value"])comparison = pd.DataFrame({    "before": values.describe(percentiles=[0.01, 0.5, 0.99]),    "after": clean.describe(percentiles=[0.01, 0.5, 0.99]),})comparison["change %"] = 100 * (comparison["after"] / comparison["before"] - 1)comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)for ax, data, title in zip(axes, [values, clean], ["Before business rules", "After business rules"]):    ax.hist(np.log10(data[data > 0]), bins=120, color="#4C72B0")    ax.set(title=title, xlabel="log10(dollar value)")axes[0].set_ylabel("transactions")plt.tight_layout()plt.show()

The body of the distribution is essentially unchanged — which is the point. Therules remove the sub-cent artefacts and the extreme tail while leaving thepopulation of real purchases intact. A cleaning step that visibly reshaped themiddle of the distribution would be destroying signal, not noise.

## 5. The fraud labels do not cover this transaction snapshotThis is the most consequential finding in this notebook and it is a schedulingproblem, not a data-cleaning one.The fraud probability files and the transaction snapshot in the repositorycover **different, non-overlapping time periods**. Every labelled fraud recordsits before the first transaction we hold. Joining them produces zero matches —not a small number, zero.

In [ ]:
consumer_fraud = spark.read.parquet(str(config.RAW_DIR / "consumer_fraud"))merchant_fraud = spark.read.parquet(str(config.RAW_DIR / "merchant_fraud"))ranges = pd.DataFrame([    {"dataset": "transactions",     "from": transactions.agg(F.min("order_datetime")).collect()[0][0],     "to": transactions.agg(F.max("order_datetime")).collect()[0][0],     "rows": transactions.count()},    {"dataset": "consumer fraud labels",     "from": consumer_fraud.agg(F.min("order_datetime")).collect()[0][0],     "to": consumer_fraud.agg(F.max("order_datetime")).collect()[0][0],     "rows": consumer_fraud.count()},    {"dataset": "merchant fraud labels",     "from": merchant_fraud.agg(F.min("order_datetime")).collect()[0][0],     "to": merchant_fraud.agg(F.max("order_datetime")).collect()[0][0],     "rows": merchant_fraud.count()},])ranges

In [ ]:
# Confirm the overlap is genuinely empty rather than merely small.matched = transactions.join(consumer_fraud, on=["user_id", "order_datetime"], how="inner")print(f"transactions matching a consumer fraud label: {matched.count()}")

### What this meansFraud labels are only useful as training data if they attach to transactions wecan compute features from. As things stand we can see *that* certain consumersand merchants were flagged, but not *what the transaction looked like* when theywere — so there is nothing to learn a model from.The fix is to obtain the earlier transaction snapshot covering the labelledperiod. The pipeline already handles this: `etl_01_raw.py` discovers snapshotfolders by glob, so dropping the second snapshot into `tables/` requires no codechange, and a `snapshot` column is carried through so the two can be told apart.Until then, the fraud model in Sprint 3 cannot be trained, and any merchant riskscore is limited to what can be inferred from transaction behaviour alone(unusual value relative to a customer's own history, sudden volume spikes).This is flagged as a project risk rather than worked around.

## 6. Summary of decisions| Issue | Extent | Decision | Why ||---|---|---|---|| Transactions with no merchant record | see section 2 | Keep, flag, exclude from ranking | No take rate or category, so unrankable; and they aren't onboarding candidates || Merchants with no transactions | small | Keep, rank separately | Explicitly the "new merchant" case in the spec || Sub-cent transaction values | see section 4 | Remove via floor at `MIN_TRANSACTION_VALUE` | Cannot be a real instalment purchase; distorts mean basket || Extreme high values | see section 4 | Threshold **per category**, not globally | A global cap is wrong in both directions across categories || Postcode to SA2 mismatch | see section 3 | Ratio-weighted attributes for modelling, dominant SA2 for maps | Avoids duplicating consumers; ABS warns postcodes are not authoritative || Fraud labels outside transaction window | total | Blocked — need the earlier snapshot | Nothing to train on until then |### Limitations to carry into the final presentation1. The postcode-to-SA2 mapping is approximate and the ABS says so. Any   demographic conclusion inherits that error.2. The transaction window is under a year, so genuine seasonality cannot be   separated from trend. Growth features should be read as short-run momentum.3. Only a small share of the consumer base transacts, so demographic features   describe the transacting population, not Australia.4. All data is synthetic. Relationships that look strong may be generator   artefacts rather than real consumer behaviour, and we should say so rather   than over-claim.

In [ ]:
spark.stop()